# Build Ground Truth Candidate Pool

This notebook builds the candidate pool used for ground-truth labeling:

1. Generate top-200 query-to-document runs for all available retrieval methods.
2. Build a top-50 candidate pool with Reciprocal Rank Fusion (RRF).
3. Shuffle and blind candidate documents before judging.

The LLM labeling step is intentionally moved to `2_llm_label_groundtruth.ipynb`.

## Notebook Setup

This setup section defines paths, constants, and project-wide assumptions for the candidate-pool build stage.

Defaults:

- `NUM_QUERIES = 500`
- `RETRIEVAL_DEPTH_PER_METHOD = 200`
- `RRF_POOL_SIZE = 50`
- `RRF_K = 60`

All build artifacts are written to `Finalproject/notebooks/rec_and_eval/groundtruth_outputs`.

In [2]:
from __future__ import annotations

import ast
import json
import math
import os
import pickle
import random
import re
import statistics
import warnings
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Iterable, Optional

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity, linear_kernel

warnings.filterwarnings("ignore")

try:
    import faiss
except ImportError:
    faiss = None

try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    SentenceTransformer = None

try:
    from rank_bm25 import BM25Okapi
except ImportError:
    BM25Okapi = None

In [3]:
def find_finalproject_root(start_path: Optional[Path] = None) -> Path:
    """Find the Finalproject directory by walking upward from the current path."""
    current_path = (start_path or Path.cwd()).resolve()
    for candidate_path in [current_path, *current_path.parents]:
        if candidate_path.name == "Finalproject":
            return candidate_path
        nested_finalproject = candidate_path / "Finalproject"
        if nested_finalproject.exists() and nested_finalproject.is_dir():
            return nested_finalproject.resolve()
    raise FileNotFoundError("Could not find the Finalproject directory.")


FINALPROJECT_ROOT = find_finalproject_root()
DATA_PATH = FINALPROJECT_ROOT / "data" / "all_recipes_final.csv"
NOTEBOOKS_ROOT = FINALPROJECT_ROOT / "notebooks"
MODELS_PATH = NOTEBOOKS_ROOT / "Saved_models"
RAREC_DIR = NOTEBOOKS_ROOT / "RA_Rec"
NOTEBOOK_OUTPUT_DIR = NOTEBOOKS_ROOT / "rec_and_eval" / "groundtruth_outputs"
METHOD_RUNS_DIR = NOTEBOOK_OUTPUT_DIR / "method_runs_top200"
POOLING_OUTPUT_DIR = NOTEBOOK_OUTPUT_DIR / "candidate_pooling"
ANNOTATION_OUTPUT_DIR = NOTEBOOK_OUTPUT_DIR / "annotation"

for directory_path in [
    NOTEBOOK_OUTPUT_DIR,
    METHOD_RUNS_DIR,
    POOLING_OUTPUT_DIR,
    ANNOTATION_OUTPUT_DIR,
]:
    directory_path.mkdir(parents=True, exist_ok=True)

In [4]:
NUM_QUERIES = 500
RETRIEVAL_DEPTH_PER_METHOD = 200
RRF_POOL_SIZE = 50
RRF_K = 60
SHUFFLE_RANDOM_SEED = 20260709

METHOD_NAMES = [
    "BM25",
    "TFIDF",
    "Ingredient_TFIDF",
    "Keyword",
    "Hybrid_Content",
    "SBERT_FAISS",
    "Hybrid_TFIDF_SBERT",
    "RARec_Late_Fusion",
]

print("Finalproject root:", FINALPROJECT_ROOT)
print("Data path:", DATA_PATH)
print("Saved models:", MODELS_PATH)
print("RARec directory:", RAREC_DIR)
print("Output directory:", NOTEBOOK_OUTPUT_DIR)

Finalproject root: C:\Users\tientm1\DS300-UIT-RecommenderSystem\Finalproject
Data path: C:\Users\tientm1\DS300-UIT-RecommenderSystem\Finalproject\data\all_recipes_final.csv
Saved models: C:\Users\tientm1\DS300-UIT-RecommenderSystem\Finalproject\notebooks\Saved_models
RARec directory: C:\Users\tientm1\DS300-UIT-RecommenderSystem\Finalproject\notebooks\RA_Rec
Output directory: C:\Users\tientm1\DS300-UIT-RecommenderSystem\Finalproject\notebooks\rec_and_eval\groundtruth_outputs


## Data Loading and Query Set Preparation

The pipeline needs two stable inputs:

1. A recipe collection with one row per recipe and a stable `doc_id`.
2. The current query file:

```text
groundtruth_outputs/queries_500_title_as_query.csv
```

The retrieval stage only requires `query_id` and `query_text`. The existing `source_doc_id` column is kept as metadata, but it is not used for inference. Every method receives the raw `query_text` and computes query-to-document scores, matching the inference notebook.

In [5]:
def load_recipe_collection(data_path: Path) -> pd.DataFrame:
    """Load recipes and guarantee a stable integer doc_id column."""
    if not data_path.exists():
        raise FileNotFoundError(f"Recipe collection not found: {data_path}")

    recipe_dataframe = pd.read_csv(data_path).reset_index(drop=True)
    if "doc_id" not in recipe_dataframe.columns:
        recipe_dataframe.insert(0, "doc_id", recipe_dataframe.index.astype(int))

    required_columns = ["doc_id", "title", "ingredients", "ingredients_normalized", "step"]
    missing_columns = [column for column in required_columns if column not in recipe_dataframe.columns]
    if missing_columns:
        raise ValueError(f"Recipe collection is missing required columns: {missing_columns}")

    for column_name in recipe_dataframe.columns:
        if recipe_dataframe[column_name].dtype == "object":
            recipe_dataframe[column_name] = recipe_dataframe[column_name].fillna("")

    return recipe_dataframe


def build_title_as_query_set(recipe_dataframe: pd.DataFrame, num_queries: int, random_seed: int) -> pd.DataFrame:
    """Build a title-sampled query file for pipeline debugging when no user query file exists."""
    if num_queries > len(recipe_dataframe):
        raise ValueError("num_queries cannot exceed the number of recipes.")

    random_generator = random.Random(random_seed)
    working_dataframe = recipe_dataframe.copy()
    working_dataframe["query_text"] = working_dataframe["title"].astype(str)

    if "type_of_food" not in working_dataframe.columns:
        sampled_queries = working_dataframe.sample(n=num_queries, random_state=random_seed)
    else:
        sampled_parts = []
        for _, category_group in working_dataframe.groupby("type_of_food", dropna=False):
            category_fraction = len(category_group) / len(working_dataframe)
            category_sample_size = max(1, round(num_queries * category_fraction))
            category_sample_size = min(category_sample_size, len(category_group))
            sampled_parts.append(
                category_group.sample(
                    n=category_sample_size,
                    random_state=random_generator.randint(0, 10**9),
                )
            )

        sampled_queries = pd.concat(sampled_parts, ignore_index=True)
        if len(sampled_queries) > num_queries:
            sampled_queries = sampled_queries.sample(n=num_queries, random_state=random_seed)
        elif len(sampled_queries) < num_queries:
            remaining_candidates = working_dataframe.loc[
                ~working_dataframe["doc_id"].isin(sampled_queries["doc_id"])
            ]
            additional_queries = remaining_candidates.sample(
                n=num_queries - len(sampled_queries),
                random_state=random_seed,
            )
            sampled_queries = pd.concat([sampled_queries, additional_queries], ignore_index=True)

    sampled_queries = sampled_queries.sample(frac=1.0, random_state=random_seed).reset_index(drop=True)
    sampled_queries = sampled_queries.rename(columns={"doc_id": "source_doc_id"})
    sampled_queries.insert(0, "query_id", range(len(sampled_queries)))

    output_columns = ["query_id", "query_text", "source_doc_id"]
    if "type_of_food" in sampled_queries.columns:
        output_columns.append("type_of_food")
    return sampled_queries[output_columns]


def load_query_set(query_path: Path, num_queries: int, random_seed: int) -> pd.DataFrame:
    """Load a query file and normalize it to query_id/query_text plus optional metadata."""
    query_dataframe = pd.read_csv(query_path)
    if "query_text" not in query_dataframe.columns:
        raise ValueError(f"Query file must contain a query_text column: {query_path}")

    query_dataframe = query_dataframe.dropna(subset=["query_text"]).copy()
    query_dataframe["query_text"] = query_dataframe["query_text"].astype(str).str.strip()
    query_dataframe = query_dataframe[query_dataframe["query_text"] != ""].reset_index(drop=True)

    if len(query_dataframe) > num_queries:
        query_dataframe = query_dataframe.sample(n=num_queries, random_state=random_seed).reset_index(drop=True)
    if len(query_dataframe) < num_queries:
        raise ValueError(
            f"Query file has only {len(query_dataframe)} valid queries, but NUM_QUERIES is {num_queries}."
        )

    if "query_id" not in query_dataframe.columns:
        query_dataframe.insert(0, "query_id", range(len(query_dataframe)))
    else:
        query_dataframe["query_id"] = query_dataframe["query_id"].astype(int)

    ordered_columns = ["query_id", "query_text"]
    optional_columns = [
        column for column in query_dataframe.columns
        if column not in ordered_columns
    ]
    return query_dataframe[ordered_columns + optional_columns]



recipes = load_recipe_collection(DATA_PATH)

# Load the current 500-query file directly.
# Retrieval still treats each row as a raw query string and does not use source_doc_id for inference.
QUERY_SET_PATH = NOTEBOOK_OUTPUT_DIR / "queries_500_title_as_query.csv"

if not QUERY_SET_PATH.exists():
    queries = build_title_as_query_set(
        recipe_dataframe=recipes,
        num_queries=NUM_QUERIES,
        random_seed=SHUFFLE_RANDOM_SEED,
    )
    queries.to_csv(QUERY_SET_PATH, index=False, encoding="utf-8-sig")

queries = load_query_set(QUERY_SET_PATH, num_queries=NUM_QUERIES, random_seed=SHUFFLE_RANDOM_SEED)

print("Number of recipes:", len(recipes))
print("Number of queries:", len(queries))
print("Query set loaded from:", QUERY_SET_PATH)
queries.head()

Number of recipes: 10263
Number of queries: 500
Query set loaded from: C:\Users\tientm1\DS300-UIT-RecommenderSystem\Finalproject\notebooks\rec_and_eval\groundtruth_outputs\queries_500_title_as_query.csv


,query_id,query_text,source_doc_id,type_of_food
0,0,Bánh trung thu nướng nhân đậu xanh bằng nồi ch...,2492,Món bánh
1,1,Hạt thuỷ tinh trong trà sữa thơm ngon đẹp mắt,9871,Trà sữa
2,2,"Chà bông (ruốc) cá ngừ thơm ngon, đơn giản tại...",5988,Món xào
3,3,Cocktail Long Island Ice Tea nổi tiếng New Yor...,9756,Thức uống
4,4,"Lẩu cù lao miền Tây thơm ngon, hấp dẫn dễ làm ...",8009,Món lẩu


## Generate Top-200 Query-to-Document Method Runs

This section generates the retrieval run files directly inside the ground-truth build pipeline. It follows the same inference direction as `recommend_and_evaluation/4_Inference.ipynb`: each method receives a raw user-style query string, applies its own training-compatible normalization, computes query-to-document scores, and writes the top 200 documents.

All method runs are saved in:

```text
groundtruth_outputs/method_runs_top200/
```

The RRF step reads these files from disk. This is deliberate: if the RRF logic changes later, the expensive model inference step does not need to be rerun.

Supported methods:

- BM25
- TF-IDF
- Ingredient TF-IDF
- Keyword Jaccard
- Hybrid content retrieval
- SBERT FAISS
- Hybrid TF-IDF + SBERT
- RARec Late Fusion

If a model artifact is missing, that method is skipped with a clear message.

In [6]:
def parse_list_string(value) -> list[str]:
    """Parse a Python-list-like field into a list of strings."""
    if pd.isna(value) or value == "[]":
        return []
    if isinstance(value, list):
        return [str(item) for item in value]
    try:
        parsed_value = ast.literal_eval(str(value))
        return [str(item) for item in parsed_value] if isinstance(parsed_value, list) else []
    except (ValueError, SyntaxError):
        return []


def parse_set_string(value) -> list[str]:
    """Parse a Python-set-like field into a list of strings."""
    if pd.isna(value) or value == "set()":
        return []
    if isinstance(value, (set, list, tuple)):
        return [str(item) for item in value]
    try:
        parsed_value = ast.literal_eval(str(value))
        if isinstance(parsed_value, (set, list, tuple)):
            return [str(item) for item in parsed_value]
        return []
    except (ValueError, SyntaxError):
        return []


def clean_text(text) -> str:
    """Apply the same simple Vietnamese text cleaning used by the training notebooks."""
    if pd.isna(text):
        return ""
    normalized_text = str(text).lower()
    normalized_text = re.sub(r"[^\w\s\u00C0-\u1EF9]", " ", normalized_text)
    normalized_text = re.sub(r"\s+", " ", normalized_text).strip()
    return normalized_text


def normalize_query_for_tfidf(query: str) -> str:
    return clean_text(query)


def normalize_query_for_ingredient_tfidf(query: str) -> str:
    return clean_text(query)


def tokenize_for_bm25(text: str) -> list[str]:
    return clean_text(text).split()


def build_bm25_document_text(row: pd.Series) -> str:
    """Build the BM25 document text from inspectable recipe fields."""
    text_parts = [
        row.get("title", ""),
        row.get("type_of_food", ""),
        row.get("description", ""),
        " ".join(parse_set_string(row.get("ingredients_normalized", ""))),
        " ".join(parse_list_string(row.get("ingredients", ""))),
        " ".join(parse_list_string(row.get("step", ""))),
    ]
    return " ".join(str(part) for part in text_parts if str(part).strip())


COOKING_METHODS = [
    "xào", "nướng", "luộc", "chiên", "hấp", "kho", "rim", "rang",
    "canh", "súp", "cháo", "gỏi", "nộm", "salad", "bún", "phở",
    "mì", "cơm", "bánh", "chè", "sinh tố",
]


def extract_keywords(title, ingredients_normalized_list=None) -> set[str]:
    """Extract query or recipe keywords with the same logic as Keyword training."""
    keywords = set()
    title_text = clean_text(title)

    for method in COOKING_METHODS:
        if method in title_text:
            keywords.add(method)

    for word in title_text.split():
        if len(word) > 2:
            keywords.add(word)

    if ingredients_normalized_list:
        for ingredient in ingredients_normalized_list:
            cleaned_ingredient = clean_text(str(ingredient))
            if len(cleaned_ingredient) > 2:
                keywords.add(cleaned_ingredient)

    return keywords


def jaccard_similarity(left_set: set[str], right_set: set[str]) -> float:
    if not left_set and not right_set:
        return 0.0
    union_size = len(left_set.union(right_set))
    if union_size == 0:
        return 0.0
    return len(left_set.intersection(right_set)) / union_size


recipes["ingredients_normalized_list"] = recipes["ingredients_normalized"].apply(parse_set_string)
recipes["keyword_set"] = recipes.apply(
    lambda row: extract_keywords(row["title"], row["ingredients_normalized_list"]),
    axis=1,
)

### Artifact Loaders and Run Writers

The loader mirrors `4_Inference.ipynb`: artifacts are loaded once, then the top-200 generation functions reuse them for all 500 queries. The run writer persists every method output as JSONL so RRF can be rerun later without recomputing retrieval scores.

In [7]:
def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


def load_pickle(path: Path):
    with path.open("rb") as file:
        return pickle.load(file)


def missing_files(paths: Iterable[Path]) -> list[str]:
    return [str(path) for path in paths if not Path(path).exists()]


def write_method_run_jsonl(
    method_name: str,
    query_dataframe: pd.DataFrame,
    ranked_docs_by_query_id: dict[int, list[dict]],
    output_directory: Path,
    depth: int,
) -> Path:
    """Write a method run JSONL file in the format consumed by RRF."""
    output_path = output_directory / f"{method_name}_top{depth}.jsonl"
    with output_path.open("w", encoding="utf-8") as output_file:
        for query_id in query_dataframe["query_id"].astype(int).tolist():
            retrieved_docs = ranked_docs_by_query_id.get(query_id, [])[:depth]
            output_record = {
                "query_id": int(query_id),
                "method_name": method_name,
                "retrieved_docs": retrieved_docs,
            }
            output_file.write(json.dumps(output_record, ensure_ascii=False) + "\n")
    print(f"Saved {method_name} run to: {output_path}")
    return output_path


def top_ranked_docs_from_scores(scores, depth: int) -> list[dict]:
    """Convert a score vector into ranked retrieval records."""
    score_array = np.asarray(scores, dtype=float)
    if len(score_array) == 0:
        return []

    finite_mask = np.isfinite(score_array)
    finite_indices = np.where(finite_mask)[0]
    if len(finite_indices) == 0:
        return []

    candidate_count = min(depth, len(finite_indices))
    finite_scores = score_array[finite_indices]
    top_positions = np.argpartition(finite_scores, -candidate_count)[-candidate_count:]
    top_doc_ids = finite_indices[top_positions]
    top_doc_ids = top_doc_ids[np.argsort(score_array[top_doc_ids])[::-1]]

    return [
        {"doc_id": int(doc_id), "rank": rank, "score": float(score_array[doc_id])}
        for rank, doc_id in enumerate(top_doc_ids, start=1)
    ]


def validate_score_length(scores, recipe_dataframe: pd.DataFrame, method_name: str) -> None:
    if len(scores) != len(recipe_dataframe):
        raise ValueError(
            f"{method_name} produced {len(scores)} scores, but recipe collection has {len(recipe_dataframe)} rows."
        )

### Load All Available Retrieval Artifacts

This cell loads artifacts for all query-time retrieval methods. Missing artifacts do not stop the notebook; they only disable the corresponding method. After rerunning the training notebooks, this audit should show most methods under `loaded_artifacts`.

In [8]:
def load_retrieval_artifacts() -> tuple[dict, dict]:
    """Load all retrieval artifacts needed for query-to-document inference."""
    loaded_artifacts = {}
    missing_artifacts = {}

    tfidf_dir = MODELS_PATH / "TFIDF"
    tfidf_required_files = [
        tfidf_dir / "tfidf_vectorizer.pkl",
        tfidf_dir / "tfidf_matrix.pkl",
        tfidf_dir / "metadata.json",
    ]
    missing = missing_files(tfidf_required_files)
    if missing:
        missing_artifacts["TFIDF"] = missing
    else:
        loaded_artifacts["TFIDF"] = {
            "vectorizer": load_pickle(tfidf_dir / "tfidf_vectorizer.pkl"),
            "matrix": load_pickle(tfidf_dir / "tfidf_matrix.pkl"),
            "metadata": load_json(tfidf_dir / "metadata.json"),
        }

    ingredient_dir = MODELS_PATH / "Ingredient_TFIDF"
    ingredient_required_files = [
        ingredient_dir / "ingredient_tfidf_vectorizer.pkl",
        ingredient_dir / "ingredient_tfidf_matrix.pkl",
        ingredient_dir / "metadata.json",
    ]
    missing = missing_files(ingredient_required_files)
    if missing:
        missing_artifacts["Ingredient_TFIDF"] = missing
    else:
        loaded_artifacts["Ingredient_TFIDF"] = {
            "vectorizer": load_pickle(ingredient_dir / "ingredient_tfidf_vectorizer.pkl"),
            "matrix": load_pickle(ingredient_dir / "ingredient_tfidf_matrix.pkl"),
            "metadata": load_json(ingredient_dir / "metadata.json"),
        }

    sbert_dir = MODELS_PATH / "SBERT_FAISS"
    sbert_required_files = [
        sbert_dir / "recipe_embeddings.npy",
        sbert_dir / "model_info.json",
    ]
    if faiss is not None:
        sbert_required_files.append(sbert_dir / "faiss_index.bin")
    missing = missing_files(sbert_required_files)
    if missing or SentenceTransformer is None:
        missing_artifacts["SBERT_FAISS"] = missing
        if SentenceTransformer is None:
            missing_artifacts["SBERT_FAISS"].append("sentence_transformers is not installed")
    else:
        model_info = load_json(sbert_dir / "model_info.json")
        loaded_artifacts["SBERT_FAISS"] = {
            "model": SentenceTransformer(model_info["model_name"]),
            "embeddings": np.load(sbert_dir / "recipe_embeddings.npy"),
            "model_info": model_info,
            "faiss_index": faiss.read_index(str(sbert_dir / "faiss_index.bin")) if faiss is not None else None,
        }

    hybrid_tfidf_sbert_dir = MODELS_PATH / "Hybrid_TFIDF_SBERT"
    hybrid_tfidf_sbert_required_files = [
        hybrid_tfidf_sbert_dir / "sbert_embeddings.npy",
        hybrid_tfidf_sbert_dir / "config.json",
    ]
    missing = missing_files(hybrid_tfidf_sbert_required_files)
    if missing:
        missing_artifacts["Hybrid_TFIDF_SBERT"] = missing
    else:
        loaded_artifacts["Hybrid_TFIDF_SBERT"] = {
            "sbert_embeddings": np.load(hybrid_tfidf_sbert_dir / "sbert_embeddings.npy"),
            "config": load_json(hybrid_tfidf_sbert_dir / "config.json"),
        }

    rarec_embeddings_path = RAREC_DIR / "recipes_embeddings_list.pkl"
    if not rarec_embeddings_path.exists() or SentenceTransformer is None:
        missing_artifacts["RARec_Late_Fusion"] = []
        if not rarec_embeddings_path.exists():
            missing_artifacts["RARec_Late_Fusion"].append(str(rarec_embeddings_path))
        if SentenceTransformer is None:
            missing_artifacts["RARec_Late_Fusion"].append("sentence_transformers is not installed")
    else:
        rarec_model = loaded_artifacts.get("SBERT_FAISS", {}).get("model")
        rarec_model_name = (
            loaded_artifacts.get("SBERT_FAISS", {})
            .get("model_info", {})
            .get("model_name", "keepitreal/vietnamese-sbert")
        )
        if rarec_model is None:
            rarec_model = SentenceTransformer(rarec_model_name)
        loaded_artifacts["RARec_Late_Fusion"] = {
            "model": rarec_model,
            "recipes_embeddings_list": load_pickle(rarec_embeddings_path),
            "model_name": rarec_model_name,
        }

    if BM25Okapi is None:
        missing_artifacts["BM25"] = ["rank_bm25 is not installed; falling back is intentionally disabled here."]

    return loaded_artifacts, missing_artifacts


loaded_artifacts, missing_artifacts = load_retrieval_artifacts()

print("Loaded artifact groups:", sorted(loaded_artifacts.keys()))
print("Missing artifact groups:", sorted(missing_artifacts.keys()))
missing_artifacts

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 43830.62it/s]


Loaded artifact groups: ['Hybrid_TFIDF_SBERT', 'Ingredient_TFIDF', 'RARec_Late_Fusion', 'SBERT_FAISS', 'TFIDF']
Missing artifact groups: ['BM25']


{'BM25': ['rank_bm25 is not installed; falling back is intentionally disabled here.']}

### Query-to-Document Top-200 Generation

Each function below accepts all 500 query rows and writes one top-200 JSONL file. The functions intentionally compute query-to-document scores, matching the inference notebook:

- TF-IDF methods use `vectorizer.transform([normalized_query])`.
- SBERT methods use `model.encode([query])`.
- RARec Late Fusion compares the query embedding against all sentence embeddings in each recipe and ranks by average similarity.

In [9]:
def generate_bm25_method_run(
    recipe_dataframe: pd.DataFrame,
    query_dataframe: pd.DataFrame,
    output_directory: Path,
    depth: int,
) -> Optional[Path]:
    if BM25Okapi is None:
        print("Skipping BM25 because rank_bm25 is not installed.")
        return None

    document_texts = recipe_dataframe.apply(build_bm25_document_text, axis=1)
    tokenized_documents = document_texts.apply(tokenize_for_bm25).tolist()
    bm25_model = BM25Okapi(tokenized_documents)

    ranked_docs_by_query_id = {}
    for _, query_row in query_dataframe.iterrows():
        query_id = int(query_row["query_id"])
        query_tokens = tokenize_for_bm25(str(query_row["query_text"]))
        scores = bm25_model.get_scores(query_tokens)
        validate_score_length(scores, recipe_dataframe, "BM25")
        ranked_docs_by_query_id[query_id] = top_ranked_docs_from_scores(scores, depth)

    return write_method_run_jsonl("BM25", query_dataframe, ranked_docs_by_query_id, output_directory, depth)


def generate_tfidf_method_run(
    recipe_dataframe: pd.DataFrame,
    query_dataframe: pd.DataFrame,
    output_directory: Path,
    depth: int,
) -> Optional[Path]:
    artifacts = loaded_artifacts.get("TFIDF")
    if artifacts is None:
        print("Skipping TFIDF because artifacts are missing.")
        return None

    ranked_docs_by_query_id = {}
    for _, query_row in query_dataframe.iterrows():
        query_id = int(query_row["query_id"])
        query_vector = artifacts["vectorizer"].transform([normalize_query_for_tfidf(query_row["query_text"])])
        scores = linear_kernel(query_vector, artifacts["matrix"]).flatten()
        validate_score_length(scores, recipe_dataframe, "TFIDF")
        ranked_docs_by_query_id[query_id] = top_ranked_docs_from_scores(scores, depth)

    return write_method_run_jsonl("TFIDF", query_dataframe, ranked_docs_by_query_id, output_directory, depth)


def generate_ingredient_tfidf_method_run(
    recipe_dataframe: pd.DataFrame,
    query_dataframe: pd.DataFrame,
    output_directory: Path,
    depth: int,
) -> Optional[Path]:
    artifacts = loaded_artifacts.get("Ingredient_TFIDF")
    if artifacts is None:
        print("Skipping Ingredient_TFIDF because artifacts are missing.")
        return None

    ranked_docs_by_query_id = {}
    for _, query_row in query_dataframe.iterrows():
        query_id = int(query_row["query_id"])
        query_vector = artifacts["vectorizer"].transform([normalize_query_for_ingredient_tfidf(query_row["query_text"])])
        scores = linear_kernel(query_vector, artifacts["matrix"]).flatten()
        validate_score_length(scores, recipe_dataframe, "Ingredient_TFIDF")
        ranked_docs_by_query_id[query_id] = top_ranked_docs_from_scores(scores, depth)

    return write_method_run_jsonl("Ingredient_TFIDF", query_dataframe, ranked_docs_by_query_id, output_directory, depth)


def generate_keyword_method_run(
    recipe_dataframe: pd.DataFrame,
    query_dataframe: pd.DataFrame,
    output_directory: Path,
    depth: int,
) -> Path:
    ranked_docs_by_query_id = {}
    recipe_keyword_sets = recipe_dataframe["keyword_set"].tolist()
    for _, query_row in query_dataframe.iterrows():
        query_id = int(query_row["query_id"])
        query_keywords = extract_keywords(query_row["query_text"], [])
        scores = np.array([
            jaccard_similarity(query_keywords, recipe_keywords)
            for recipe_keywords in recipe_keyword_sets
        ])
        validate_score_length(scores, recipe_dataframe, "Keyword")
        ranked_docs_by_query_id[query_id] = top_ranked_docs_from_scores(scores, depth)

    return write_method_run_jsonl("Keyword", query_dataframe, ranked_docs_by_query_id, output_directory, depth)


def generate_hybrid_content_method_run(
    recipe_dataframe: pd.DataFrame,
    query_dataframe: pd.DataFrame,
    output_directory: Path,
    depth: int,
    tfidf_weight: float = 0.4,
    ingredient_tfidf_weight: float = 0.6,
) -> Optional[Path]:
    tfidf_artifacts = loaded_artifacts.get("TFIDF")
    ingredient_artifacts = loaded_artifacts.get("Ingredient_TFIDF")
    if tfidf_artifacts is None or ingredient_artifacts is None:
        print("Skipping Hybrid_Content because TFIDF or Ingredient_TFIDF artifacts are missing.")
        return None

    total_weight = tfidf_weight + ingredient_tfidf_weight
    ranked_docs_by_query_id = {}
    for _, query_row in query_dataframe.iterrows():
        query_id = int(query_row["query_id"])
        tfidf_query = tfidf_artifacts["vectorizer"].transform([normalize_query_for_tfidf(query_row["query_text"])])
        ingredient_query = ingredient_artifacts["vectorizer"].transform([normalize_query_for_ingredient_tfidf(query_row["query_text"])])
        tfidf_scores = linear_kernel(tfidf_query, tfidf_artifacts["matrix"]).flatten()
        ingredient_scores = linear_kernel(ingredient_query, ingredient_artifacts["matrix"]).flatten()
        scores = (tfidf_weight / total_weight) * tfidf_scores + (ingredient_tfidf_weight / total_weight) * ingredient_scores
        validate_score_length(scores, recipe_dataframe, "Hybrid_Content")
        ranked_docs_by_query_id[query_id] = top_ranked_docs_from_scores(scores, depth)

    return write_method_run_jsonl("Hybrid_Content", query_dataframe, ranked_docs_by_query_id, output_directory, depth)

In [10]:
def generate_sbert_faiss_method_run(
    recipe_dataframe: pd.DataFrame,
    query_dataframe: pd.DataFrame,
    output_directory: Path,
    depth: int,
) -> Optional[Path]:
    artifacts = loaded_artifacts.get("SBERT_FAISS")
    if artifacts is None:
        print("Skipping SBERT_FAISS because artifacts are missing.")
        return None

    ranked_docs_by_query_id = {}
    model = artifacts["model"]
    faiss_index = artifacts.get("faiss_index")
    embeddings = artifacts["embeddings"]

    for _, query_row in query_dataframe.iterrows():
        query_id = int(query_row["query_id"])
        query_embedding = model.encode(
            [str(query_row["query_text"])],
            convert_to_numpy=True,
            normalize_embeddings=True,
        )

        if faiss_index is not None:
            scores, indices = faiss_index.search(query_embedding, depth)
            ranked_docs_by_query_id[query_id] = [
                {"doc_id": int(doc_id), "rank": rank, "score": float(score)}
                for rank, (doc_id, score) in enumerate(zip(indices[0], scores[0]), start=1)
                if int(doc_id) >= 0
            ]
        else:
            scores = cosine_similarity(query_embedding, embeddings).flatten()
            validate_score_length(scores, recipe_dataframe, "SBERT_FAISS")
            ranked_docs_by_query_id[query_id] = top_ranked_docs_from_scores(scores, depth)

    return write_method_run_jsonl("SBERT_FAISS", query_dataframe, ranked_docs_by_query_id, output_directory, depth)


def generate_hybrid_tfidf_sbert_method_run(
    recipe_dataframe: pd.DataFrame,
    query_dataframe: pd.DataFrame,
    output_directory: Path,
    depth: int,
) -> Optional[Path]:
    tfidf_artifacts = loaded_artifacts.get("TFIDF")
    sbert_artifacts = loaded_artifacts.get("SBERT_FAISS")
    hybrid_artifacts = loaded_artifacts.get("Hybrid_TFIDF_SBERT")
    if tfidf_artifacts is None or sbert_artifacts is None or hybrid_artifacts is None:
        print("Skipping Hybrid_TFIDF_SBERT because TFIDF, SBERT, or hybrid artifacts are missing.")
        return None

    alpha = float(hybrid_artifacts["config"].get("alpha", 0.5))
    model = sbert_artifacts["model"]
    sbert_embeddings = hybrid_artifacts["sbert_embeddings"]
    ranked_docs_by_query_id = {}

    for _, query_row in query_dataframe.iterrows():
        query_id = int(query_row["query_id"])
        query_text = str(query_row["query_text"])
        tfidf_query = tfidf_artifacts["vectorizer"].transform([normalize_query_for_tfidf(query_text)])
        tfidf_scores = linear_kernel(tfidf_query, tfidf_artifacts["matrix"]).flatten()
        query_embedding = model.encode([query_text], convert_to_numpy=True, normalize_embeddings=True)
        sbert_scores = cosine_similarity(query_embedding, sbert_embeddings).flatten()
        scores = alpha * tfidf_scores + (1.0 - alpha) * sbert_scores
        validate_score_length(scores, recipe_dataframe, "Hybrid_TFIDF_SBERT")
        ranked_docs_by_query_id[query_id] = top_ranked_docs_from_scores(scores, depth)

    return write_method_run_jsonl("Hybrid_TFIDF_SBERT", query_dataframe, ranked_docs_by_query_id, output_directory, depth)


def generate_rarec_late_fusion_method_run(
    recipe_dataframe: pd.DataFrame,
    query_dataframe: pd.DataFrame,
    output_directory: Path,
    depth: int,
) -> Optional[Path]:
    artifacts = loaded_artifacts.get("RARec_Late_Fusion")
    if artifacts is None:
        print("Skipping RARec_Late_Fusion because artifacts are missing.")
        return None

    model = artifacts["model"]
    recipes_embeddings_list = artifacts["recipes_embeddings_list"]
    if len(recipes_embeddings_list) != len(recipe_dataframe):
        raise ValueError(
            f"RARec embedding list has {len(recipes_embeddings_list)} rows, but recipes has {len(recipe_dataframe)} rows."
        )

    ranked_docs_by_query_id = {}
    for _, query_row in query_dataframe.iterrows():
        query_id = int(query_row["query_id"])
        query_embedding = model.encode([str(query_row["query_text"])], convert_to_numpy=True)
        query_norm = np.linalg.norm(query_embedding)
        if query_norm == 0:
            ranked_docs_by_query_id[query_id] = []
            continue
        query_embedding = query_embedding / query_norm

        scores = np.full(len(recipe_dataframe), -np.inf, dtype=float)
        for recipe_idx, dish_embeddings in enumerate(recipes_embeddings_list):
            if dish_embeddings is None or len(dish_embeddings) == 0:
                continue
            dish_embeddings = np.asarray(dish_embeddings)
            norms = np.linalg.norm(dish_embeddings, axis=1, keepdims=True)
            norms[norms == 0] = 1.0
            normalized_dish_embeddings = dish_embeddings / norms
            similarities = np.dot(normalized_dish_embeddings, query_embedding.T).flatten()
            scores[recipe_idx] = float(np.mean(similarities))

        ranked_docs_by_query_id[query_id] = top_ranked_docs_from_scores(scores, depth)

    return write_method_run_jsonl("RARec_Late_Fusion", query_dataframe, ranked_docs_by_query_id, output_directory, depth)

### Generate and Persist All Top-200 Runs

Run this cell once after the saved models are available. It overwrites the method JSONL files for the current query set. RRF reads these files from disk in the next step.

In [11]:
def generate_all_available_method_runs(
    recipe_dataframe: pd.DataFrame,
    query_dataframe: pd.DataFrame,
    output_directory: Path,
    depth: int,
) -> list[Path]:
    """Generate top-k query-to-document run files for all available methods."""
    generation_functions = [
        generate_bm25_method_run,
        generate_tfidf_method_run,
        generate_ingredient_tfidf_method_run,
        generate_keyword_method_run,
        generate_hybrid_content_method_run,
        generate_sbert_faiss_method_run,
        generate_hybrid_tfidf_sbert_method_run,
        generate_rarec_late_fusion_method_run,
    ]

    generated_paths = []
    for generation_function in generation_functions:
        output_path = generation_function(
            recipe_dataframe=recipe_dataframe,
            query_dataframe=query_dataframe,
            output_directory=output_directory,
            depth=depth,
        )
        if output_path is not None:
            generated_paths.append(output_path)

    print(f"Generated {len(generated_paths)} method run files.")
    return generated_paths


# Run this cell to generate or refresh all persisted top-200 method runs.
generated_method_run_paths = generate_all_available_method_runs(
    recipe_dataframe=recipes,
    query_dataframe=queries,
    output_directory=METHOD_RUNS_DIR,
    depth=RETRIEVAL_DEPTH_PER_METHOD,
)
generated_method_run_paths

Skipping BM25 because rank_bm25 is not installed.
Saved TFIDF run to: C:\Users\tientm1\DS300-UIT-RecommenderSystem\Finalproject\notebooks\rec_and_eval\groundtruth_outputs\method_runs_top200\TFIDF_top200.jsonl
Saved Ingredient_TFIDF run to: C:\Users\tientm1\DS300-UIT-RecommenderSystem\Finalproject\notebooks\rec_and_eval\groundtruth_outputs\method_runs_top200\Ingredient_TFIDF_top200.jsonl
Saved Keyword run to: C:\Users\tientm1\DS300-UIT-RecommenderSystem\Finalproject\notebooks\rec_and_eval\groundtruth_outputs\method_runs_top200\Keyword_top200.jsonl
Saved Hybrid_Content run to: C:\Users\tientm1\DS300-UIT-RecommenderSystem\Finalproject\notebooks\rec_and_eval\groundtruth_outputs\method_runs_top200\Hybrid_Content_top200.jsonl
Saved SBERT_FAISS run to: C:\Users\tientm1\DS300-UIT-RecommenderSystem\Finalproject\notebooks\rec_and_eval\groundtruth_outputs\method_runs_top200\SBERT_FAISS_top200.jsonl
Saved Hybrid_TFIDF_SBERT run to: C:\Users\tientm1\DS300-UIT-RecommenderSystem\Finalproject\notebook

[WindowsPath('C:/Users/tientm1/DS300-UIT-RecommenderSystem/Finalproject/notebooks/rec_and_eval/groundtruth_outputs/method_runs_top200/TFIDF_top200.jsonl'),
 WindowsPath('C:/Users/tientm1/DS300-UIT-RecommenderSystem/Finalproject/notebooks/rec_and_eval/groundtruth_outputs/method_runs_top200/Ingredient_TFIDF_top200.jsonl'),
 WindowsPath('C:/Users/tientm1/DS300-UIT-RecommenderSystem/Finalproject/notebooks/rec_and_eval/groundtruth_outputs/method_runs_top200/Keyword_top200.jsonl'),
 WindowsPath('C:/Users/tientm1/DS300-UIT-RecommenderSystem/Finalproject/notebooks/rec_and_eval/groundtruth_outputs/method_runs_top200/Hybrid_Content_top200.jsonl'),
 WindowsPath('C:/Users/tientm1/DS300-UIT-RecommenderSystem/Finalproject/notebooks/rec_and_eval/groundtruth_outputs/method_runs_top200/SBERT_FAISS_top200.jsonl'),
 WindowsPath('C:/Users/tientm1/DS300-UIT-RecommenderSystem/Finalproject/notebooks/rec_and_eval/groundtruth_outputs/method_runs_top200/Hybrid_TFIDF_SBERT_top200.jsonl'),
 WindowsPath('C:/Users/

## Step 1. Candidate Pooling with Reciprocal Rank Fusion

RRF consumes the persisted `top 200` run files generated above. Each method run is stored as JSONL in `method_runs_top200/`:

```json
{"query_id": 0, "method_name": "BM25", "retrieved_docs": [{"doc_id": 123, "rank": 1, "score": 12.34}]}
```

RRF combines ranks, not raw scores, because sparse, dense, and hybrid retrieval scores are usually not on the same scale.

```text
RRF_score(d) = sum_i 1 / (RRF_K + rank_i(d))
```

With `RRF_K = 60`, the fusion is stable and does not over-reward a single rank-1 appearance too aggressively. Each method retrieves top 200 documents for all 500 queries, and RRF keeps the top 50 fused documents per query to control annotation cost.

RRF reads run files from disk. It does not depend on in-memory variables from the retrieval generation cells.

In [12]:
@dataclass(frozen=True)
class RetrievedDocument:
    """One retrieved document from one method for one query."""
    doc_id: int
    rank: int
    score: Optional[float] = None


@dataclass(frozen=True)
class MethodRunRecord:
    """One method ranking for one query."""
    query_id: int
    method_name: str
    retrieved_docs: list[RetrievedDocument]


def load_jsonl_records(jsonl_path: Path) -> list[dict]:
    """Load a JSONL file into a list of dictionaries."""
    records = []
    with jsonl_path.open("r", encoding="utf-8") as input_file:
        for line in input_file:
            if line.strip():
                records.append(json.loads(line))
    return records


def load_method_run_jsonl(run_path: Path) -> list[MethodRunRecord]:
    """Load one method run JSONL file."""
    records = []
    for raw_record in load_jsonl_records(run_path):
        retrieved_docs = [
            RetrievedDocument(
                doc_id=int(raw_doc["doc_id"]),
                rank=int(raw_doc.get("rank", index + 1)),
                score=None if raw_doc.get("score") is None else float(raw_doc["score"]),
            )
            for index, raw_doc in enumerate(raw_record["retrieved_docs"])
        ]
        records.append(
            MethodRunRecord(
                query_id=int(raw_record["query_id"]),
                method_name=str(raw_record["method_name"]),
                retrieved_docs=retrieved_docs,
            )
        )
    return records


def discover_method_run_files(method_runs_directory: Path, depth: int) -> list[Path]:
    """Discover persisted method run files for the configured retrieval depth."""
    return sorted(method_runs_directory.glob(f"*_top{depth}.jsonl"))


def load_all_method_runs_from_files(method_runs_directory: Path, depth: int) -> dict[str, dict[int, list[RetrievedDocument]]]:
    """Load all available method runs from persisted JSONL files."""
    all_runs = {}
    run_paths = discover_method_run_files(method_runs_directory, depth)
    if not run_paths:
        raise FileNotFoundError(
            f"No method run files found in {method_runs_directory}. "
            "Run the top-200 generation cell before RRF."
        )

    for run_path in run_paths:
        method_records = load_method_run_jsonl(run_path)
        if not method_records:
            continue
        method_name = method_records[0].method_name
        all_runs[method_name] = {record.query_id: record.retrieved_docs for record in method_records}
        print(f"Loaded {len(method_records):4d} query rankings for {method_name} from {run_path.name}")
    return all_runs


def validate_method_runs(
    method_runs_by_method: dict[str, dict[int, list[RetrievedDocument]]],
    query_ids: Iterable[int],
    expected_depth: int,
) -> pd.DataFrame:
    """Create an audit table describing coverage and depth for each loaded method."""
    query_id_list = list(query_ids)
    query_id_set = set(query_id_list)
    audit_rows = []
    for method_name, method_runs in method_runs_by_method.items():
        covered_queries = set(method_runs.keys())
        depth_values = [len(method_runs[query_id]) for query_id in covered_queries if query_id in query_id_set]
        minimum_depth = min(depth_values) if depth_values else 0
        median_depth = statistics.median(depth_values) if depth_values else 0
        missing_query_count = len(query_id_set - covered_queries)
        audit_rows.append(
            {
                "method_name": method_name,
                "covered_queries": len(covered_queries & query_id_set),
                "missing_queries": missing_query_count,
                "minimum_depth": minimum_depth,
                "median_depth": median_depth,
                "expected_depth": expected_depth,
                "is_ready_for_pooling": missing_query_count == 0 and minimum_depth >= expected_depth,
            }
        )
    return pd.DataFrame(audit_rows).sort_values("method_name").reset_index(drop=True)


method_runs_by_method = load_all_method_runs_from_files(METHOD_RUNS_DIR, RETRIEVAL_DEPTH_PER_METHOD)
method_run_audit = validate_method_runs(
    method_runs_by_method=method_runs_by_method,
    query_ids=queries["query_id"].tolist(),
    expected_depth=RETRIEVAL_DEPTH_PER_METHOD,
)
method_run_audit

Loaded  500 query rankings for Hybrid_Content from Hybrid_Content_top200.jsonl
Loaded  500 query rankings for Hybrid_TFIDF_SBERT from Hybrid_TFIDF_SBERT_top200.jsonl
Loaded  500 query rankings for Ingredient_TFIDF from Ingredient_TFIDF_top200.jsonl
Loaded  500 query rankings for Keyword from Keyword_top200.jsonl
Loaded  500 query rankings for RARec_Late_Fusion from RARec_Late_Fusion_top200.jsonl
Loaded  500 query rankings for SBERT_FAISS from SBERT_FAISS_top200.jsonl
Loaded  500 query rankings for TFIDF from TFIDF_top200.jsonl


,method_name,covered_queries,missing_queries,minimum_depth,median_depth,expected_depth,is_ready_for_pooling
0,Hybrid_Content,500,0,200,200.0,200,True
1,Hybrid_TFIDF_SBERT,500,0,200,200.0,200,True
2,Ingredient_TFIDF,500,0,200,200.0,200,True
3,Keyword,500,0,200,200.0,200,True
4,RARec_Late_Fusion,500,0,200,200.0,200,True
5,SBERT_FAISS,500,0,200,200.0,200,True
6,TFIDF,500,0,200,200.0,200,True


In [13]:
def reciprocal_rank_fusion_for_query(
    query_id: int,
    method_runs_by_method: dict[str, dict[int, list[RetrievedDocument]]],
    rrf_k: int,
    retrieval_depth_per_method: int,
    pool_size: int,
) -> list[dict]:
    """Fuse method rankings for one query and return the top pooled candidates."""
    fused_scores = defaultdict(float)
    source_ranks_by_doc_id = defaultdict(dict)

    for method_name, method_runs in method_runs_by_method.items():
        ranked_documents = method_runs.get(query_id, [])[:retrieval_depth_per_method]
        for fallback_rank, retrieved_document in enumerate(ranked_documents, start=1):
            rank = retrieved_document.rank or fallback_rank
            doc_id = retrieved_document.doc_id
            fused_scores[doc_id] += 1.0 / (rrf_k + rank)
            source_ranks_by_doc_id[doc_id][method_name] = rank

    sorted_candidates = sorted(fused_scores.items(), key=lambda item: (-item[1], item[0]))[:pool_size]
    pooled_candidates = []
    for pooled_rank, (doc_id, rrf_score) in enumerate(sorted_candidates, start=1):
        source_ranks = dict(source_ranks_by_doc_id[doc_id])
        pooled_candidates.append(
            {
                "query_id": int(query_id),
                "doc_id": int(doc_id),
                "rrf_rank": int(pooled_rank),
                "rrf_score": float(rrf_score),
                "appeared_in_method_count": int(len(source_ranks)),
                "source_ranks": source_ranks,
            }
        )
    return pooled_candidates


def build_rrf_candidate_pools(
    query_dataframe: pd.DataFrame,
    method_runs_by_method: dict[str, dict[int, list[RetrievedDocument]]],
    rrf_k: int,
    retrieval_depth_per_method: int,
    pool_size: int,
) -> list[dict]:
    """Build RRF pools for all queries."""
    all_pooled_candidates = []
    for query_id in query_dataframe["query_id"].astype(int).tolist():
        all_pooled_candidates.extend(
            reciprocal_rank_fusion_for_query(
                query_id=query_id,
                method_runs_by_method=method_runs_by_method,
                rrf_k=rrf_k,
                retrieval_depth_per_method=retrieval_depth_per_method,
                pool_size=pool_size,
            )
        )
    return all_pooled_candidates


pooled_candidates = build_rrf_candidate_pools(
    query_dataframe=queries,
    method_runs_by_method=method_runs_by_method,
    rrf_k=RRF_K,
    retrieval_depth_per_method=RETRIEVAL_DEPTH_PER_METHOD,
    pool_size=RRF_POOL_SIZE,
)
pooled_candidates_path = POOLING_OUTPUT_DIR / "rrf_candidate_pool_top50.jsonl"
with pooled_candidates_path.open("w", encoding="utf-8") as output_file:
    for candidate in pooled_candidates:
        output_file.write(json.dumps(candidate, ensure_ascii=False) + "\n")

print("Number of pooled candidates:", len(pooled_candidates))
print("Saved candidate pool to:", pooled_candidates_path)

Number of pooled candidates: 25000
Saved candidate pool to: C:\Users\tientm1\DS300-UIT-RecommenderSystem\Finalproject\notebooks\rec_and_eval\groundtruth_outputs\candidate_pooling\rrf_candidate_pool_top50.jsonl


## Step 2. Shuffle Candidate Documents Before Judging

The top-50 candidates for each query are shuffled before annotation. The annotation input must not reveal:

- original rank from any method,
- original retrieval score,
- method name that retrieved the document,
- RRF score,
- RRF rank.

Those fields remain in the pooling audit file, but they should never be shown to the LLM judge or human annotators. The judging input contains only the query text and recipe content.

In [14]:
def parse_list_like_field(value) -> list[str]:
    """Parse fields stored as Python-list-like strings."""
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return [str(item) for item in value]
    text_value = str(value).strip()
    if not text_value or text_value == "[]":
        return []
    try:
        parsed_value = ast.literal_eval(text_value)
        if isinstance(parsed_value, (list, tuple, set)):
            return [str(item) for item in parsed_value]
    except (SyntaxError, ValueError):
        pass
    return [text_value]


def compact_text_list(items: list[str], max_items: int, max_characters: int) -> str:
    """Convert a list of text fragments into a compact annotation-friendly string."""
    selected_items = [str(item).strip() for item in items if str(item).strip()][:max_items]
    joined_text = " | ".join(selected_items)
    if len(joined_text) > max_characters:
        joined_text = joined_text[: max_characters - 3].rstrip() + "..."
    return joined_text


def build_recipe_lookup(recipe_dataframe: pd.DataFrame) -> dict[int, dict]:
    """Build a doc_id -> recipe metadata lookup for annotation input creation."""
    lookup = {}
    for _, row in recipe_dataframe.iterrows():
        doc_id = int(row["doc_id"])
        ingredients = parse_list_like_field(row.get("ingredients", ""))
        normalized_ingredients = parse_list_like_field(row.get("ingredients_normalized", ""))
        cooking_steps = parse_list_like_field(row.get("step", ""))
        lookup[doc_id] = {
            "doc_id": doc_id,
            "recipe_title": str(row.get("title", "")),
            "recipe_type": str(row.get("type_of_food", "")),
            "recipe_description": str(row.get("description", "")),
            "ingredients": compact_text_list(ingredients, max_items=20, max_characters=1600),
            "normalized_ingredients": compact_text_list(normalized_ingredients, max_items=20, max_characters=1200),
            "cooking_steps": compact_text_list(cooking_steps, max_items=8, max_characters=2200),
            "source_url": str(row.get("link", "")),
        }
    return lookup


def create_blinded_annotation_items(
    pooled_candidates: list[dict],
    query_dataframe: pd.DataFrame,
    recipe_lookup: dict[int, dict],
    random_seed: int,
) -> list[dict]:
    """Create shuffled query-document annotation items without rank or method metadata."""
    query_lookup = {
        int(row["query_id"]): {
            "query_text": str(row["query_text"]),
        }
        for _, row in query_dataframe.iterrows()
    }
    candidates_by_query_id = defaultdict(list)
    for candidate in pooled_candidates:
        candidates_by_query_id[int(candidate["query_id"])].append(candidate)

    random_generator = random.Random(random_seed)
    annotation_items = []
    for query_id, query_candidates in sorted(candidates_by_query_id.items()):
        shuffled_candidates = list(query_candidates)
        random_generator.shuffle(shuffled_candidates)
        for blinded_position, candidate in enumerate(shuffled_candidates, start=1):
            doc_id = int(candidate["doc_id"])
            recipe = recipe_lookup[doc_id]
            annotation_items.append(
                {
                    "query_id": int(query_id),
                    "doc_id": doc_id,
                    "blinded_position": int(blinded_position),
                    "query_text": query_lookup[query_id]["query_text"],
                    "recipe_title": recipe["recipe_title"],
                    "recipe_type": recipe["recipe_type"],
                    "recipe_description": recipe["recipe_description"],
                    "ingredients": recipe["ingredients"],
                    "normalized_ingredients": recipe["normalized_ingredients"],
                    "cooking_steps": recipe["cooking_steps"],
                }
            )
    return annotation_items


recipe_lookup = build_recipe_lookup(recipes)
pooled_candidates = load_jsonl_records(POOLING_OUTPUT_DIR / "rrf_candidate_pool_top50.jsonl")
blinded_annotation_items = create_blinded_annotation_items(
    pooled_candidates=pooled_candidates,
    query_dataframe=queries,
    recipe_lookup=recipe_lookup,
    random_seed=SHUFFLE_RANDOM_SEED,
)

blinded_annotation_items_path = ANNOTATION_OUTPUT_DIR / "blinded_annotation_items.jsonl"
with blinded_annotation_items_path.open("w", encoding="utf-8") as output_file:
    for annotation_item in blinded_annotation_items:
        output_file.write(json.dumps(annotation_item, ensure_ascii=False) + "\n")

print("Number of annotation items:", len(blinded_annotation_items))
print("Saved blinded annotation items to:", blinded_annotation_items_path)

Number of annotation items: 25000
Saved blinded annotation items to: C:\Users\tientm1\DS300-UIT-RecommenderSystem\Finalproject\notebooks\rec_and_eval\groundtruth_outputs\annotation\blinded_annotation_items.jsonl


## Manual Check: Map Shuffled `doc_id` Values Back to Recipes

This check uses the shuffled annotation file, not the RRF file. It samples query IDs, then maps each candidate `doc_id` back to the recipe table and displays the real title and compact recipe content.

Use this before running a large LLM labeling job. If IDs are shifted, the `actual_title` will not match the `recipe_title` stored in the annotation item.

In [15]:
MANUAL_CHECK_RANDOM_SEED = 20260709
MANUAL_CHECK_NUM_QUERIES = 10
MANUAL_CHECK_DOCS_PER_QUERY = 10


def compact_text(items: list[str], max_items: int = 8, max_characters: int = 500) -> str:
    selected_items = [str(item).strip() for item in items if str(item).strip()][:max_items]
    joined_text = " | ".join(selected_items)
    if len(joined_text) > max_characters:
        joined_text = joined_text[: max_characters - 3].rstrip() + "..."
    return joined_text


def build_recipe_id_lookup(recipe_dataframe: pd.DataFrame) -> dict[int, dict]:
    lookup = {}
    for _, row in recipe_dataframe.iterrows():
        doc_id = int(row["doc_id"])
        lookup[doc_id] = {
            "actual_title": str(row.get("title", "")),
            "actual_type": str(row.get("type_of_food", "")),
            "actual_ingredients": compact_text(parse_list_like_field(row.get("ingredients", ""))),
            "actual_normalized_ingredients": compact_text(parse_list_like_field(row.get("ingredients_normalized", ""))),
            "actual_steps": compact_text(parse_list_like_field(row.get("step", "")), max_items=4, max_characters=700),
            "actual_link": str(row.get("link", "")),
        }
    return lookup


def build_manual_id_mapping_check(
    annotation_items: list[dict],
    recipe_dataframe: pd.DataFrame,
    num_queries: int,
    docs_per_query: int,
    random_seed: int,
) -> pd.DataFrame:
    annotation_dataframe = pd.DataFrame(annotation_items)
    if annotation_dataframe.empty:
        raise ValueError("No blinded annotation items were loaded.")

    recipe_lookup = build_recipe_id_lookup(recipe_dataframe)
    available_query_ids = sorted(annotation_dataframe["query_id"].astype(
        int).unique().tolist())
    random_generator = random.Random(random_seed)
    sampled_query_ids = random_generator.sample(
        available_query_ids,
        k=min(num_queries, len(available_query_ids)),
    )

    rows = []
    for query_id in sampled_query_ids:
        query_items = (
            annotation_dataframe.loc[annotation_dataframe["query_id"].astype(int) == int(query_id)]
            .sort_values("blinded_position")
            .head(docs_per_query)
        )
        for _, item in query_items.iterrows():
            doc_id = int(item["doc_id"])
            actual_recipe = recipe_lookup.get(doc_id, {})
            rows.append({
                "query_id": int(query_id),
                "query_text": item["query_text"],
                "blinded_position": int(item["blinded_position"]),
                "doc_id": doc_id,
                "annotation_title": item.get("recipe_title", ""),
                "actual_title": actual_recipe.get("actual_title", ""),
                "title_match": item.get("recipe_title", "") == actual_recipe.get("actual_title", ""),
                "actual_type": actual_recipe.get("actual_type", ""),
                "actual_normalized_ingredients": actual_recipe.get("actual_normalized_ingredients", ""),
                "actual_steps": actual_recipe.get("actual_steps", ""),
                "actual_link": actual_recipe.get("actual_link", ""),
            })
    return pd.DataFrame(rows)


manual_id_mapping_check = build_manual_id_mapping_check(
    annotation_items=blinded_annotation_items,
    recipe_dataframe=recipes,
    num_queries=MANUAL_CHECK_NUM_QUERIES,
    docs_per_query=MANUAL_CHECK_DOCS_PER_QUERY,
    random_seed=MANUAL_CHECK_RANDOM_SEED,
)

print("Rows shown:", len(manual_id_mapping_check))
print("All sampled title mappings match:", bool(manual_id_mapping_check["title_match"].all()))
manual_id_mapping_check

Rows shown: 100
All sampled title mappings match: True


,query_id,query_text,blinded_position,doc_id,annotation_title,actual_title,title_match,actual_type,actual_normalized_ingredients,actual_steps,actual_link
0,440,Bánh khoai tây chiên giòn nhân thịt phô mai ta...,1,7028,"Bánh tart khoai tây chiên giòn rụm, đơn giản c...","Bánh tart khoai tây chiên giòn rụm, đơn giản c...",True,Món chiên,muối/ tiêu xay | dầu ăn | bột thì là | bột mì ...,Bước 1: Sơ chế nguyên liệu: Khoai tây bạn gọt ...,https://www.dienmayxanh.com/vao-bep/cach-lam-b...
1,440,Bánh khoai tây chiên giòn nhân thịt phô mai ta...,2,9024,Khoai tây lốc xoáy ăn ngon bá cháy,Khoai tây lốc xoáy ăn ngon bá cháy,True,Ăn vặt,khoai tây | bột phô mai | dầu ăn,Bước 1: Sơ chế nguyên liệu: Khoai tây rửa sạch...,https://www.dienmayxanh.com/vao-bep/cach-lam-k...
2,440,Bánh khoai tây chiên giòn nhân thịt phô mai ta...,3,754,Vào bếp làm món khoai tây lồng đèn vừa xinh vừ...,Vào bếp làm món khoai tây lồng đèn vừa xinh vừ...,True,Nhanh và dễ,khoai tây | tương ớt chai | bột chiên giòn bịc...,Bước 1: Sơ chế khoai tây Khoai tây mua về bạn ...,https://vncooking.com/cong-thuc/vao-bep-lam-mo...
3,440,Bánh khoai tây chiên giòn nhân thịt phô mai ta...,4,2502,Bánh mì tròn Bagel nhân khoai tây phô mai béo ...,Bánh mì tròn Bagel nhân khoai tây phô mai béo ...,True,Món bánh,đường | bơ lạt | muối | men khô | khoai tây | ...,Bước 1: Trộn bột bánh: Cho vào tô 200g bột bán...,https://www.dienmayxanh.com/vao-bep/cach-lam-b...
4,440,Bánh khoai tây chiên giòn nhân thịt phô mai ta...,5,2726,Bánh khoai tây chiên giòn nhân thịt phô mai ta...,Bánh khoai tây chiên giòn nhân thịt phô mai ta...,True,Món bánh,thịt nạc heo xay | chén bột mì | bột chiên xù ...,Bước 1: Trộn bột bánh và gia vị: Cho 50g bột n...,https://www.dienmayxanh.com/vao-bep/cach-lam-b...
...,...,...,...,...,...,...,...,...,...,...,...
95,344,"Bánh thuyền Đài Loan nhỏ bằng lò nướng ngon, đ...",6,4745,Cách nướng khoai bằng lò vi sóng siêu nhanh đơ...,Cách nướng khoai bằng lò vi sóng siêu nhanh đơ...,True,Món nướng,khoai lang,Bước 1: Sơ chế khoai lang: Chọn mua khoai lang...,https://www.dienmayxanh.com/vao-bep/cach-nuong...
96,344,"Bánh thuyền Đài Loan nhỏ bằng lò nướng ngon, đ...",7,2338,Bánh trung thu dẻo vị bí đỏ thơm ngon bổ dưỡng...,Bánh trung thu dẻo vị bí đỏ thơm ngon bổ dưỡng...,True,Món bánh,nước đường nướng bánh | dầu ăn | bột bánh dẻo ...,Bước 1: Hấp và nghiền bí: Bí đỏ sau khi mua về...,https://www.dienmayxanh.com/vao-bep/cach-lam-b...
97,344,"Bánh thuyền Đài Loan nhỏ bằng lò nướng ngon, đ...",8,730,Bánh nướng nhân sữa dừa thơm ngon,Bánh nướng nhân sữa dừa thơm ngon,True,Các loại bánh,vỏ bánh | vani | nước đường | đường tùy chọn |...,"Bước 1: Ướp dừa tươi nạo sợi với sữa đặc, đườn...",https://vnexpress.net/doi-song-cooking-banh-nu...
98,344,"Bánh thuyền Đài Loan nhỏ bằng lò nướng ngon, đ...",9,3704,"Bánh nhãn (bánh bi, cà) thơm ngon giòn rụm ăn ...","Bánh nhãn (bánh bi, cà) thơm ngon giòn rụm ăn ...",True,Món bánh,đường | trứng gà | bột nếp | gừng tươi | nước,Bước 1: Trộn bột: Đập trứng vào tô rồi đánh ta...,https://www.dienmayxanh.com/vao-bep/cach-lam-b...


In [16]:
for query_id, query_group in manual_id_mapping_check.groupby("query_id", sort=False):
    query_text = query_group["query_text"].iloc[0]
    print("=" * 120)
    print(f"query_id={query_id} | query={query_text}")
    display(
        query_group[[
            "blinded_position",
            "doc_id",
            "title_match",
            "actual_title",
            "actual_type",
            "actual_normalized_ingredients",
        ]]
    )

query_id=440 | query=Bánh khoai tây chiên giòn nhân thịt phô mai tan chảy


,blinded_position,doc_id,title_match,actual_title,actual_type,actual_normalized_ingredients
0,1,7028,True,"Bánh tart khoai tây chiên giòn rụm, đơn giản c...",Món chiên,muối/ tiêu xay | dầu ăn | bột thì là | bột mì ...
1,2,9024,True,Khoai tây lốc xoáy ăn ngon bá cháy,Ăn vặt,khoai tây | bột phô mai | dầu ăn
2,3,754,True,Vào bếp làm món khoai tây lồng đèn vừa xinh vừ...,Nhanh và dễ,khoai tây | tương ớt chai | bột chiên giòn bịc...
3,4,2502,True,Bánh mì tròn Bagel nhân khoai tây phô mai béo ...,Món bánh,đường | bơ lạt | muối | men khô | khoai tây | ...
4,5,2726,True,Bánh khoai tây chiên giòn nhân thịt phô mai ta...,Món bánh,thịt nạc heo xay | chén bột mì | bột chiên xù ...
5,6,982,True,Bánh gạo Hàn Quốc lắc phô mai,Món tráng miệng,bột phô mai | bánh gạo
6,7,7000,True,Khoai tây chiên bằng nồi chiên không dầu giòn ...,Món chiên,kg khoai tây | muối | dầu ăn | bột mì đa dụng
7,8,2844,True,Bánh mì phô mai sữa Hạ Long bằng nồi chiên khô...,Món bánh,phô mai mozzarella | bơ lạt | phô mai con bò c...
8,9,3739,True,Chả giò khoai tây thơm ngon giòn rụm đơn giản ...,Món bánh,bắp mỹ | khoai tây khoảng | dầu ăn | bánh trán...
9,10,2605,True,Bánh khoai tây Aligot phô mai thơm ngon béo ng...,Món bánh,heavy cream | phô mai mozzarella | bột chiên x...


query_id=248 | query=Cá vược hấp bia ngon ngọt, lạ miệng đơn giản tại nhà


,blinded_position,doc_id,title_match,actual_title,actual_type,actual_normalized_ingredients
10,1,7105,True,"Cá đuối chiên sả ớt thơm ngon, hấp dẫn ai ăn c...",Món chiên,gia vị thông dụng muối/ đường/ tiêu/ bột ngọt ...
11,2,7830,True,"Cá vược hấp xì dầu bằng xửng hấp inox đẹp mắt,...",Món hấp,gừng | hành boa rô | đường | dầu hoa cải | muố...
12,3,7758,True,"Cá bớp hấp nấm linh chi thơm ngon, lạ miệng đơ...",Món hấp,gừng | nấm linh chi tươi | hành lá | át cá bớp...
13,4,4572,True,Cá thòi lòi nướng muối ớt ngon miệng đậm đà lạ...,Món nướng,cá thòi lòi | muối | ớt
14,5,4790,True,Cá rô nướng thơm ngon hấp dẫn đơn giản dễ làm ...,Món nướng,muối hột | muối/ bột ngọt | cá rô | ớt | dầu dừa
15,6,4585,True,Cá bò da nướng giấy bạc thơm ngon hấp dẫn ngay...,Món nướng,dầu ăn | muối/hạt nêm | rau xà lách | sả | cá ...
16,7,10075,True,"Khô cá kèo thơm ngon, đậm vị đơn giản tại nhà",Món khô - mắm,cá kèo | ớt | tỏi | gia vị thông dụng muối/ bộ...
17,8,4700,True,"Cá basa nướng muối ớt thơm lừng, cay cay mềm n...",Món nướng,mỡ hành | cá basa khoảng | gia vị thông dụng m...
18,9,9601,True,Bia matcha kiểu nhật thơm mát hội chị em thích mê,Thức uống,muỗng cà phê bột trà xanh matcha | nước nóng |...
19,10,7264,True,Cá mương chiên giòn rụm bằng chảo nhôm cho bữa...,Món chiên,bột chiên giòn | cá mương | rượu trắng và muối...


query_id=178 | query=Bánh flan táo không bị rổ, lạ miệng, béo mịn


...

,blinded_position,doc_id,title_match,actual_title,actual_type,actual_normalized_ingredients
70,1,5271,True,"Su hào xào tỏi giòn thơm, thanh vị cho bữa ăn ...",Món xào,gia vị thông dụng hạt nêm/ đường/ bột ngọt/ ti...
71,2,5277,True,Rau xuyến chi xào tỏi xanh giòn thơm ngon bằng...,Món xào,gia vị thông dụng dầu ăn/ muối/ đường/ bột ngọ...
72,3,4768,True,Cách nướng ngô bằng lò nướng cực ngon hấp dẫn ...,Món nướng,dầu ăn | bơ | muối | ngô ngon bắp | hành lá
73,4,10123,True,Gừng ngâm giấm để dùng dần rất đơn giản mà hữu...,Món khô - mắm,đường cát | muối | gừng tươi | giấm giấm táo/g...
74,5,7263,True,Cách ướp cốt lết chiên không bị khô mềm ngon s...,Món chiên,hành tím | thịt đùi sườn cốt lết | gia vị thôn...
75,6,4755,True,Cách ướp gà nướng truyền thống đơn giản mà ai ...,Món nướng,đường | bột nêm | tiêu tùy khẩu vị | dầu ăn | ...
76,7,8113,True,Chi tiết Caprese salad đơn giản tại nhà với mo...,Món gỏi - salad,phô mai mozzarella | húng quế | chai iấm balsa...
77,8,9200,True,Nui chiên bơ tỏi giòn tan đậm vị ăn vặt cực thích,Ăn vặt,dầu ăn | nui | tương ớt | tỏi băm | gia vị thô...
78,9,8193,True,Caesar salad là gì? Cách làm chi tiết caesar s...,Món gỏi - salad,sốt worcestershire | phô mai parmesan | trứng ...
79,10,5358,True,"Đậu rồng xào tỏi thơm lừng, giòn sựt cực đưa cơm",Món xào,gia vị thông dụng hạt nêm/ muối/ bột ngọt | dầ...


query_id=216 | query=Giảm cân sau Tết với món salad bắp cải giòn ngon đúng điệu


,blinded_position,doc_id,title_match,actual_title,actual_type,actual_normalized_ingredients
80,1,8379,True,"Kimchi bắp cải giòn ngon, ăn cùng đồ nướng cực...",Món gỏi - salad,đường | hành tây | gừng băm | nước mắm | muối ...
81,2,458,True,"Bắp cải cuốn thịt mềm ngon, đậm vị",Món ngon ngày lạnh,hạt tiêu | nạc vai xay | muối | gừng nhỏ | bắp...
82,3,5899,True,"Bắp cải xào thịt heo thơm ngon, đơn giản cho c...",Món xào,gia vị thông dụng muối/ hạt nêm/ tiêu xay | dầ...
83,4,8617,True,Bắp cải cuộn nấm chay thanh đạm đổi món cho bữ...,Món chay,nấm bạch tuyết | dầu ăn | muối | bắp cải xanh ...
84,5,5333,True,Bắp cải xào thịt hộp lạ miệng đổi vị bữa cơm v...,Món xào,gia vị thông dụng nước mắm/ hạt nêm/ đường/ ti...
85,6,8172,True,"Gỏi tai heo bắp cải giòn ngon, đơn giản tại nhà",Món gỏi - salad,rau răm | hành tím | chanh | đậu phộng rang | ...
86,7,8134,True,Salad trứng cá hồi giàu dinh dưỡng ăn hoài khô...,Món gỏi - salad,mayonnaise | bắp cải trắng | mì tảo bẹ | thanh...
87,8,1360,True,Canh bắp cải thịt bằm đơn giản thơm ngon ngọt ...,Món canh,muỗng canh bột canh | dầu ăn | hành tím | hành...
88,9,769,True,"Kimbap bắp cải đậu hũ đơn giản, thanh đạm, bổ ...",Món chính,trái ớt | chén gạo lứt | miếng đậu hũ trắng | ...
89,10,5792,True,"Bắp cải xào thịt băm thơm ngon, cực nhanh chón...",Món xào,gia vị thông dụng muối/ hạt nêm/ bột ngọt/ đườ...


query_id=344 | query=Bánh thuyền Đài Loan nhỏ bằng lò nướng ngon, đãi khách ngày Tết


,blinded_position,doc_id,title_match,actual_title,actual_type,actual_normalized_ingredients
90,1,2652,True,"Bánh thuyền Đài Loan nhỏ bằng lò nướng ngon, đ...",Món bánh,hạnh nhân lát | mè đen | đường | nam việt quất...
91,2,2935,True,Bánh chuối nướng bằng lò nướng đơn giản ngon h...,Món bánh,sữa tươi | đường | dầu ăn | baking soda | chuố...
92,3,2378,True,Bánh trung thu nướng nhân cốm dừa thơm ngon dễ...,Món bánh,dừa nạo | bột mì số | bơ lạt | lòng đỏ trứng m...
93,4,8870,True,Cách nướng khoai bằng thìa inox ngon như nướng...,Ăn vặt,khoai lang | cái inox
94,5,3351,True,Bánh gạo rong biển giòn từ cơm nguội bằng lò v...,Món bánh,dầu ăn | cơm | mật ong | xì dầu | á rong biển
95,6,4745,True,Cách nướng khoai bằng lò vi sóng siêu nhanh đơ...,Món nướng,khoai lang
96,7,2338,True,Bánh trung thu dẻo vị bí đỏ thơm ngon bổ dưỡng...,Món bánh,nước đường nướng bánh | dầu ăn | bột bánh dẻo ...
97,8,730,True,Bánh nướng nhân sữa dừa thơm ngon,Các loại bánh,vỏ bánh | vani | nước đường | đường tùy chọn |...
98,9,3704,True,"Bánh nhãn (bánh bi, cà) thơm ngon giòn rụm ăn ...",Món bánh,đường | trứng gà | bột nếp | gừng tươi | nước
99,10,2132,True,Bánh Macaron không cần lò nướng thơm ngon bất bại,Món bánh,bột hạnh nhân | màu thực phẩm xanh dương/tím |...


## Candidate-Pool Build Checklist

File 1 should produce and preserve these artifacts:

- `queries_500_title_as_query.csv`
- one top-200 run file per retrieval method in `method_runs_top200/`
- `candidate_pooling/rrf_candidate_pool_top50.jsonl`
- `annotation/blinded_annotation_items.jsonl`

The next notebook, `2_llm_label_groundtruth.ipynb`, loads the shuffled annotation file and assigns LLM relevance labels.